# Day 12 練習：Context 是怎麼被一層一層放上工作桌的？

這份練習不比較模型答案，也不需要呼叫 Qwen。你要看的只有一件事：**每一步，系統又放了什麼資料給模型看？**

## Goal

執行到最後時，你應能說出：使用者只打了一句旅行需求，但模型眼前還有規則、偏好、前面的對話、可用工具說明與工具結果。這些資料不是模型自己記起來的，而是 Harness 在這一輪放進 Context。

## Setup

所有資料都是固定的教學情境；天氣結果是模擬資料，不是即時查詢。這裡的 `source` 只是幫人類辨認資料從哪裡來。

請由上到下執行一次；若要重新開始，重新執行第一個程式格來清空工作桌。

In [1]:
import json

context = []  # 重新執行此格即可清空工作桌

def add_to_desk(role, source, content, why):
    item = {"role": role, "source": source, "content": content}
    context.append(item)
    print(f"\n剛加入：{source}")
    print(f"內容：{content}")
    print(f"用途：{why}")
    print(f"目前工作桌共有 {len(context)} 份資料。")

def show_desk():
    print("\n=== 最後完整 Context ===")
    for number, item in enumerate(context, start=1):
        print(f"{number}. [{item['source']}] {item['content']}")

### 第 1 步：系統規則

系統在使用者開口以前，就可能先放入工作規則。這不是使用者打的字。

In [2]:
add_to_desk(
    "system",
    "系統規則",
    "你是旅行助理。行程要標示交通方式與可能的天氣風險。",
    "限制回答的角色與基本工作方式。",
)


剛加入：系統規則
內容：你是旅行助理。行程要標示交通方式與可能的天氣風險。
用途：限制回答的角色與基本工作方式。
目前工作桌共有 1 份資料。


### 第 2 步：虛擬記憶

假設產品先前保存了使用者偏好，並決定這一輪仍然相關，所以重新放到工作桌。模型不是自己永久記住它。

In [3]:
add_to_desk(
    "system",
    "虛擬記憶",
    "使用者偏好大眾運輸，不喜歡一天換很多次交通工具。",
    "讓模型安排交通時能考慮使用者偏好。",
)


剛加入：虛擬記憶
內容：使用者偏好大眾運輸，不喜歡一天換很多次交通工具。
用途：讓模型安排交通時能考慮使用者偏好。
目前工作桌共有 2 份資料。


### 第 3 步：Skill 說明

系統也可以告訴模型手上有哪些能力。Skill 只說明「可以做什麼」，還沒有提供天氣答案；它是產品定義的能力說明，不是模型內建的魔法。

In [4]:
add_to_desk(
    "system",
    "Skill 說明",
    "weather_tool 可以查詢指定日期與城市的天氣。資料不足時先詢問日期。",
    "讓模型知道它能用什麼工具補資料。",
)


剛加入：Skill 說明
內容：weather_tool 可以查詢指定日期與城市的天氣。資料不足時先詢問日期。
用途：讓模型知道它能用什麼工具補資料。
目前工作桌共有 3 份資料。


## Steps

### 第 0 步：使用者的第一句話

使用者只說：「請安排台北三天旅行，步調不要太趕。」此刻模型只知道任務本身；它還不知道使用者偏好什麼交通，也不知道天氣。

In [5]:
add_to_desk(
    "user",
    "第一輪使用者請求",
    "請安排台北三天旅行，步調不要太趕。",
    "告訴模型這一輪要完成的任務。",
)


剛加入：第一輪使用者請求
內容：請安排台北三天旅行，步調不要太趕。
用途：告訴模型這一輪要完成的任務。
目前工作桌共有 4 份資料。


### 第 4 步：多一輪對話

助手先追問日期，使用者補上「下週五到週日」。前面說過的話會成為下一輪的對話歷史。

執行後，工作桌會從 4 份變成 6 份資料。

In [6]:
context[3]["source"] = "對話歷史（第一輪使用者）"
add_to_desk(
    "assistant",
    "對話歷史",
    "請問三天是哪幾天？我會先確認天氣。",
    "保留前一輪助手為何追問。",
)
add_to_desk(
    "user",
    "目前使用者請求",
    "下週五到週日；如果第一天下午下雨，安排室內景點。",
    "補上日期與新的行程條件。",
)


剛加入：對話歷史
內容：請問三天是哪幾天？我會先確認天氣。
用途：保留前一輪助手為何追問。
目前工作桌共有 5 份資料。

剛加入：目前使用者請求
內容：下週五到週日；如果第一天下午下雨，安排室內景點。
用途：補上日期與新的行程條件。
目前工作桌共有 6 份資料。


### Agent 呼叫工具

使用者補上日期後，Agent 決定呼叫 `weather_tool`。這一步還沒有天氣答案；它只是把查詢工作交給外部工具。

In [7]:
print("Agent：呼叫 weather_tool，查詢台北下週五天氣。")

Agent：呼叫 weather_tool，查詢台北下週五天氣。


### 第 6 步：工具結果回到工作桌

外部程式查完天氣後，必須把結果放回 Context，模型才看得到。Skill 是能力說明；下面這則才是這次查到的結果。

In [8]:
add_to_desk(
    "tool",
    "工具結果",
    "weather_tool（模擬，非即時天氣）：下週五台北午後可能有短暫陣雨。",
    "讓模型能根據實際查詢結果安排週五下午。",
)


剛加入：工具結果
內容：weather_tool（模擬，非即時天氣）：下週五台北午後可能有短暫陣雨。
用途：讓模型能根據實際查詢結果安排週五下午。
目前工作桌共有 7 份資料。


## 最後：看看模型的工作桌

執行下一格。你看到的不是『模型自動知道的一切』，而是這個示範系統在此刻選擇交給它的所有資料。

下方 JSON 只供好奇的讀者查看，不必讀懂。

In [9]:
show_desk()

model_messages = [{"role": item["role"], "content": item["content"]} for item in context]
print("\n給模型的訊息格式（移除教學用 source 標籤）：")
print(json.dumps(model_messages, ensure_ascii=False, indent=2))


=== 最後完整 Context ===
1. [系統規則] 你是旅行助理。行程要標示交通方式與可能的天氣風險。
2. [虛擬記憶] 使用者偏好大眾運輸，不喜歡一天換很多次交通工具。
3. [Skill 說明] weather_tool 可以查詢指定日期與城市的天氣。資料不足時先詢問日期。
4. [對話歷史（第一輪使用者）] 請安排台北三天旅行，步調不要太趕。
5. [對話歷史] 請問三天是哪幾天？我會先確認天氣。
6. [目前使用者請求] 下週五到週日；如果第一天下午下雨，安排室內景點。
7. [工具結果] weather_tool（模擬，非即時天氣）：下週五台北午後可能有短暫陣雨。

給模型的訊息格式（移除教學用 source 標籤）：
[
  {
    "role": "system",
    "content": "你是旅行助理。行程要標示交通方式與可能的天氣風險。"
  },
  {
    "role": "system",
    "content": "使用者偏好大眾運輸，不喜歡一天換很多次交通工具。"
  },
  {
    "role": "system",
    "content": "weather_tool 可以查詢指定日期與城市的天氣。資料不足時先詢問日期。"
  },
  {
    "role": "user",
    "content": "請安排台北三天旅行，步調不要太趕。"
  },
  {
    "role": "assistant",
    "content": "請問三天是哪幾天？我會先確認天氣。"
  },
  {
    "role": "user",
    "content": "下週五到週日；如果第一天下午下雨，安排室內景點。"
  },
  {
    "role": "tool",
    "content": "weather_tool（模擬，非即時天氣）：下週五台北午後可能有短暫陣雨。"
  }
]


## Takeaway

你最該觀察的不是程式碼，而是每一格的「剛加入」：

- 目前請求與前面對話：要做什麼、前面發生過什麼。
- 系統規則：怎麼做。
- 虛擬記憶：使用者過去偏好什麼。
- Skill：可以用什麼能力。
- 對話歷史：前面發生過什麼。
- 工具結果：剛剛查到什麼。

這些資料合在一起，才是模型這一輪的 Context。不同產品會放入不同內容，也可能摘要或刪除舊資料。